In [ ]:
# SEGMENTATION OF CONDENSATES

# Key steps of the workflow:
# Min-max intensity normalization
# 2D Gaussian smoothing (slice-by-slice)
# 3D spot filter to detect dots
# Watershed for seperating falsely merged dots
import os
import pandas as pd
import numpy as np
import sys
from nd2reader import ND2Reader
from cellpose import models
from tifffile import imwrite, imread, TiffFile
import matplotlib.pyplot as plt
import cv2
import glob
from glob import glob
from skimage import io
from skimage import feature, exposure, draw, morphology, filters, img_as_ubyte, measure, segmentation
from skimage.morphology import remove_small_objects,  dilation, erosion, ball     
from skimage.segmentation import watershed
from skimage.measure import label
from skimage import (
    io,
    filters,
    measure,
    segmentation,
    morphology,
    exposure,
    draw,
    feature,
    transform,
    img_as_ubyte
)
from scipy import ndimage
from scipy.stats import norm
from scipy.ndimage import gaussian_filter
import scipy.ndimage as ndi
from scipy.ndimage import gaussian_filter, gaussian_laplace
from scipy.ndimage import distance_transform_edt
from stardist.data import test_image_nuclei_2d
from stardist.plot import render_label
from csbdeep.utils import normalize
from csbdeep.utils import Path, normalize

# Function definitions: Adapted from: allen cell structure segmenter dot segmentation workflow
# https://github.com/AllenCell/aics-segmentation

In [ ]:
def intensity_normalization(struct_img, scaling_param):

    '''
    Mode 1:  scaling_param = [0]
    Mode 2:  scaling_param = [lower std range, upper std range]
    Mode 3:  scaling_param = [lower std range, upper std range, lower abs intensity, higher abs intensity]
    '''
    assert len(scaling_param) > 0

    if len(scaling_param) == 1:
        if scaling_param[0] < 1:
            #print('intensity normalization: using min-max normalization with NO absolute intensity upper bound')
            pass
        else:
            #print(f'intensity normalization: using min-max normalization with absolute intensity upper bound {scaling_param[0]}')
            struct_img[struct_img > scaling_param[0]] = struct_img.min()
        strech_min = struct_img.min()
        strech_max = struct_img.max()
        struct_img = (struct_img - strech_min + 1e-8)/(strech_max - strech_min + 1e-8)
    elif len(scaling_param) == 2:
        # print(f'intensity normalization: normalize into [mean - {scaling_param[0]} x std, mean + {scaling_param[1]} x std] ')
        m, s = norm.fit(struct_img.flat)
        # print(m,s)
        # import numpy as np
        # import pdb; pdb.set_trace()
        strech_min = max(m - scaling_param[0] * s, struct_img.min())
        strech_max = min(m + scaling_param[1] * s, struct_img.max())
        struct_img[struct_img > strech_max] = strech_max
        struct_img[struct_img < strech_min] = strech_min
        struct_img = (struct_img - strech_min + 1e-8)/(strech_max - strech_min + 1e-8)
    elif len(scaling_param) == 4:
        img_valid = struct_img[np.logical_and(struct_img > scaling_param[2], struct_img < scaling_param[3])]
        m, s = norm.fit(img_valid.flat)
        strech_min = max(scaling_param[2] - scaling_param[0] * s, struct_img.min())
        strech_max = min(scaling_param[3] + scaling_param[1] * s, struct_img.max())
        struct_img[struct_img > strech_max] = strech_max
        struct_img[struct_img < strech_min] = strech_min
        struct_img = (struct_img - strech_min + 1e-8)/(strech_max - strech_min + 1e-8)

    # print('intensity normalization completes')
    return struct_img

def image_smoothing_gaussian(struct_img, sigma, truncate_range=3.0):
    structure_img_smooth = gaussian_filter(struct_img, sigma=sigma, mode='nearest', truncate=truncate_range)
    return structure_img_smooth

def image_smoothing_gaussian_slice_by_slice(struct_img, sigma, truncate_range=3.0):
    """
    wrapper for applying 2D Guassian smoothing slice by slice on a 3D image
    """
    structure_img_smooth = np.zeros_like(struct_img)
    for zz in range(struct_img.shape[0]):
        structure_img_smooth[zz, :, :] = gaussian_filter(
            struct_img[zz, :, :], sigma=sigma, mode="nearest", truncate=truncate_range
        )

    return structure_img_smooth

def dot_3d_wrapper(struct_img, s3_param):
    """wrapper for 3D spot filter

    Parameters:
    ------------
    struct_img: np.ndarray
        a 3d numpy array, usually the image after smoothing
    s3_param: List
        [[scale_1, cutoff_1], [scale_2, cutoff_2], ....], e.g. [[1, 0.1]]
        or [[1,0.12], [3,0.1]]. scale_x is set based on the estimated radius of
        your target dots. For example, if visually the diameter of the dots is
        about 3~4 pixels, then you may want to set scale_x as 1 or something
        near 1 (like 1.25). Multiple scales can be used, if you have dots of
        very different sizes. cutoff_x is a threshold applied on the actual filter
        reponse to get the binary result. Smaller cutoff_x may yielf more dots and
        "fatter" segmentation, while larger cutoff_x could be less permisive and
        yield less dots and slimmer segmentation.
    """

    bw = np.zeros(struct_img.shape, dtype=bool)
    for fid in range(len(s3_param)):
        log_sigma = s3_param[fid][0]
        responce = -1 * (log_sigma**2) * gaussian_laplace(struct_img, log_sigma)
        bw = np.logical_or(bw, responce > s3_param[fid][1])
    return bw

def peak_local_max_wrapper(struct_img_for_peak, bw):
    from skimage.feature import peak_local_max

    local_maxi = peak_local_max(struct_img_for_peak, labels=label(bw), min_distance=2)
    local_maxi_image = np.zeros_like(struct_img_for_peak)
    local_maxi_image[tuple(local_maxi.T)] = True
    return local_maxi_image

def detect_spots(test_img_cond, sm_sigma=1, spot_params=[[1, 0.05]], ws=False, minArea = 1, maxArea=None):
    # Normalization: apply once to entire image 
    test_img_cond_norm = intensity_normalization(test_img_cond, scaling_param=[0])
    
    # Gaussian smoothing, slice by slice
    test_img_cond_norm_smooth = image_smoothing_gaussian_slice_by_slice(test_img_cond_norm, sigma=sm_sigma)
    # Spot detection
    # spot detection parameters: first value is the sigma for gaussian, second is threshold
    # multiple sigma values can be provided; larger sigma values will detect larger spots
    # decrease threshold to detect more  spots
    # e.g.: spot_params = [[1, 0.04], [2, 0.06]]
    # return value is a (boolean) mask image
    bw = dot_3d_wrapper(test_img_cond_norm_smooth, spot_params)

    # watershed to separate any touching spots.  Any spots less than minArea pixels will be removed.  
    # Done before watershed, for efficiency, and then after once again
    if ws:
        Mask = remove_small_objects(bw, min_size=minArea, connectivity=1) 
        Seed = dilation(peak_local_max_wrapper(test_img_cond_norm, Mask), footprint=ball(1))
        Watershed_Map = -1*distance_transform_edt(bw)
        seg = watershed(Watershed_Map, label(Seed), mask=Mask, watershed_line=True)
        seg = remove_small_objects(seg, min_size=minArea, connectivity=1)
        # Apply 3D connected component labeling to mark spots over zlevels
        lseg = label(seg)
    else:
        Mask = remove_small_objects(bw, min_size=minArea, connectivity=1) 
        # Apply 3D connected component labeling to mark spots over zlevels
        lseg = label(Mask)

    # Filter by **maxArea** if provided
    if maxArea is not None:
        props = measure.regionprops(lseg)
        for region in props:
            if region.area > maxArea:
                lseg[lseg == region.label] = 0
        # Relabel after removing large objects
        lseg = label(lseg)
    
    return lseg

def make_annotated_stack(orig_img, labeled_stack1, labeled_stack2, bit_depth=16):
    # Label each spot with its id on each image slice, and outline it as well
    # second input labeled stack is the dapi, the ids for dapi are not printed only the outlines

    dtype_max = (2**bit_depth)-1
    font = cv2.FONT_HERSHEY_SIMPLEX  
    font_scale = 0.25  
    thickness = 1
    
    labels1 = np.zeros_like(orig_img)
    labels2 = np.zeros_like(orig_img)
    for z in range(labeled_stack1.shape[0]):  # Iterate through z-slices
        labels1[z] = segmentation.mark_boundaries(labels1[z], labeled_stack1[z], color=(dtype_max, 0, 0) , mode='thick')[:,:,0]
        labels2[z] = segmentation.mark_boundaries(labels2[z], labeled_stack2[z], color=(dtype_max, 0, 0) , mode='thick')[:,:,0]
        
        region_props = measure.regionprops(labeled_stack1[z])
        for region in region_props:
            y, x = map(int, region.centroid)  
            labels1[z] = cv2.putText(labels1[z], str(region.label), (x, y), 
                                     font, font_scale, dtype_max, thickness, cv2.LINE_AA)

    return np.stack([orig_img, labels1, labels2], axis=1)

def detect_background(orig_img, labeled_stack, th_method):
    # For each z level, apply threshold, and remove any pixels that are objects from labeled mask
    # then, get background mean intensity

    # Dynamically construct the function name
    threshold_func = getattr(filters, f"threshold_{th_method}")

    bk_img_stack = []
    bk_intens = []
    for z in range(orig_img.shape[0]):  
        th = threshold_func(orig_img[z])
        bk_mask = orig_img[z] <= th
        bk_mask = bk_mask * (labeled_stack[z] == 0)

        bk_img = orig_img[z] * bk_mask
        bk_intens.append([z,np.mean(bk_img.flatten())])
        bk_img_stack.append(bk_mask)

    return bk_intens, np.array(bk_img_stack)
        

In [ ]:
###TEST THE PARAMETERS HERE###

test_img = io.imread("")  # Shape: ZYXC
file_root = ""
output_dir = ""

# Define parameters
mid_Z = 17
condensate_channel = 5
dapi_channel = 0
bit_depth = 16
XY_micron_per_px = 0.1833
Z_micron_per_px = 0.5

use_custom_model = False  # << Toggle this
custom_model_path = ''  # << Only used if above is True

print(test_img.shape) #Should be (Z, X, Y, C)

In [ ]:
# Assuming testimg is (Z, C, X, Y) and should be (Z, X, Y, C)
test_img = np.transpose(test_img, (0, 2, 3, 1))  # Swap axes
#0 (Z) stays in place.
#2 (X) moves to the second position.
#3 (Y) moves to the third position.
#1 (C) moves to the fourth position.
print("New shape:", test_img.shape)  # Should print (Z, X, Y, C)

In [ ]:
# === Model Selection ===
if use_custom_model:
    model = models.CellposeModel(gpu=False, pretrained_model=custom_model_path)
    model_is_3D = False
else:
    model = models.Cellpose(gpu=False, model_type='nuclei')
    model_is_3D = True

use_dapi = True
clear_border = True
bk_th_method = 'otsu'
sm_sigma = 1

param1 = 1
param2 = 0.07 #smaller numbers = more detected
spot_params = [[param1, param2]]

ws = False
minArea = 1
maxArea = 1000

scale_factor = 0.6
est_nucl_diam = 120

# Extract channels
condensate_img = test_img[..., condensate_channel]
dapi_img = test_img[..., dapi_channel]

# Detect condensates
lmask_condensates = detect_spots(condensate_img, sm_sigma=sm_sigma, spot_params=spot_params, ws=ws,
                                 minArea=minArea, maxArea=maxArea)

# Segment nuclei
cp_dapi_img = transform.rescale(dapi_img, scale=[1, scale_factor, scale_factor])
cp_dapi_img = normalize(cp_dapi_img, 1, 99.8, axis=(0, 1, 2))

# === Segmentation ===
if model_is_3D:
    # Use 3D segmentation
    dapi_labels, flows, styles, diams = model.eval(
        cp_dapi_img,
        channels=[0, 0],
        diameter=int(est_nucl_diam * scale_factor),
        do_3D = True,
        anisotropy=2.73
    )
    
    # Resize 3D mask to original image shape
    dapi_labels = transform.resize(dapi_labels, dapi_img.shape, order=0, preserve_range=True)
    dapi_labels = dapi_labels.astype(np.uint16)

else:
    # Slice-by-slice 2D segmentation
    dapi_labels = np.zeros_like(dapi_img, dtype=np.uint16)
    
    for z in range(dapi_img.shape[0]):
        slice_img = cp_dapi_img[z]
        masks, flows, styles = model.eval(
            slice_img,
            channels=[0, 0],
            diameter=int(est_nucl_diam * scale_factor),
            do_3D= False
        )
        
        fullres_mask = transform.resize(
            masks,
            dapi_img.shape[1:],
            order=0,
            preserve_range=True
        ).astype(np.uint16)
        
        dapi_labels[z] = fullres_mask

if clear_border:
    dapi_labels = segmentation.clear_border(dapi_labels)

In [ ]:
#Show the images:

plt.figure(figsize=(15, 4))
# Original condensate image
plt.subplot(1, 3, 1)
plt.imshow(condensate_img[mid_Z], cmap='gray')
plt.title(f"Condensate Image (Z={mid_Z})")
plt.axis('off')

# Condensate label mask
plt.subplot(1, 3, 2)
plt.imshow(lmask_condensates[mid_Z], cmap='turbo')
plt.title(f"Condensate Labels (Z={mid_Z})")
plt.axis('off')

# DAPI nuclear labels
plt.subplot(1, 3, 3)
plt.imshow(dapi_labels[mid_Z], cmap='nipy_spectral')
plt.title(f"Nuclear Labels (Z={mid_Z})")
plt.axis('off')

plt.tight_layout()
plt.show()


In [ ]:
# Save test output image
ann_img = make_annotated_stack(condensate_img, lmask_condensates, dapi_labels, bit_depth=bit_depth)
imwrite(os.path.join(output_dir, f"{file_root}_{param1}-{param2}.tif"), ann_img, imagej=True, metadata={'axes': 'ZCYX'})

In [ ]:
# Setup...
# Note: this script works for images in this format: [z,x,y] (z channel first)
base_dir = ""
images_dir = f"{base_dir}/Tifs"
output_dir = f"{base_dir}/Results_detect-condensates"

condensate_channel = 5
dapi_channel = 0
bit_depth = 16  # 16 bit image
Z_micron_per_px = 0.5
XY_micron_per_px = 0.1833

# Foci detection parameters
sm_sigma=1  # smoothing amount prior to detection
spot_params=[[1, 0.07]]  # detection params
ws=False  # whether to use watershed to separate 'touching' spots, probably not good to use in this case
minArea = 1  # min area in px for counting a spot
maxArea = 1000

use_custom_model = True  # << Toggle this
custom_model_path = ''  # << Only used if above is True
# === Model Selection ===
if use_custom_model:
    model = models.CellposeModel(gpu=False, pretrained_model=custom_model_path)
    model_is_3D = False
else:
    model = models.Cellpose(gpu=False, model_type='nuclei')
    model_is_3D = True

# which threshold to use to detect the condensate channel background level
# case-sensitive, must match one of the skimage.filters threshold options: mean,minimum,otsu,yen,triangle
bk_th_method = 'otsu' 

# Should we additionally make sure that no cell nuclei are included when calculating background?
# I set to false bc I think that some nucleus might not exhibit condensates so could be considered background
# Change if needed
use_dapi = True

# should we count foci per nuclei for the nuclei touching border?
# or remove them?
clear_border = True

# Cell pose: factor to scale down the image + est nucl diameter (px)
scale_factor = 0.6 
est_nucl_diam = 100

# contrast enhance for output annotated image (raise for more CE)
ce_saturate_perc=0

In [ ]:
# BEGIN...
# DO NOT FORGET TO UN-COMMENT THE RESHAPING LINE IF RESHAPING IS NEEDED!!!!!

image_files = glob(f"{images_dir}/*.tif")
foci_data = pd.DataFrame()
nucleus_data = pd.DataFrame()
bk_data = pd.DataFrame()
for file_ in image_files:
    file_root = os.path.splitext(os.path.split(file_)[1])[0]
    print(file_root)
    
    # Load image in original (check if ZYXC) format
    next_img = io.imread(file_)  # sometimes alt shape: (Z, C, Y, X)
    
    # Reshape to ZYXC if needed
    next_img = np.transpose(next_img, (0, 2, 3, 1))  # new shape: (Z, Y, X, C)
    print(next_img.shape) #confidence check (should always be Z, Y, X, C)
    
    condensate_img = next_img[:,:,:,condensate_channel]
    dapi_img = next_img[:,:,:,dapi_channel]

    # detect all condensates...
    lmask_condensates = detect_spots(condensate_img, 
                                     sm_sigma=sm_sigma, 
                                     spot_params=spot_params, 
                                     ws=ws, 
                                     minArea=minArea,
                                     maxArea=maxArea)

    # detect nuclei w cellpose
    # resize down first
    cp_dapi_img = transform.rescale(dapi_img, scale = [1, scale_factor, scale_factor])

    # normalize image
    axis_norm=(0, 1, 2)
    cp_dapi_img = normalize(cp_dapi_img, 1, 99.8, axis=axis_norm)

    # === Segmentation ===
    if model_is_3D:
        # Use 3D segmentation
        dapi_labels, flows, styles, diams = model.eval(
            cp_dapi_img,
            channels=[0, 0],
            diameter=int(est_nucl_diam * scale_factor),
            do_3D=True
        )
        
        # Resize 3D mask to original image shape
        dapi_labels = transform.resize(dapi_labels, dapi_img.shape, order=0, preserve_range=True)
        dapi_labels = dapi_labels.astype(np.uint16)

    else:
        # Slice-by-slice 2D segmentation
        dapi_labels = np.zeros_like(dapi_img, dtype=np.uint16)
        
        for z in range(dapi_img.shape[0]):
            slice_img = cp_dapi_img[z]
            masks, flows, styles = model.eval(
                slice_img,
                channels=[0, 0],
                diameter=int(est_nucl_diam * scale_factor),
                do_3D=False
            )
            
            fullres_mask = transform.resize(
                masks,
                dapi_img.shape[1:],
                order=0,
                preserve_range=True
            ).astype(np.uint16)
            
            dapi_labels[z] = fullres_mask

    # Returns background intensity at each z level, which is found by thresholding. 
    # Threshold method can be adjusted by user (default Otsu).  If any areas detected as 
    # nuclei are included in background after thresholding, those will also be removed.
    if use_dapi:
        dapi_labels_in = dapi_labels
    else:
        dapi_labels_in = np.zeros_like(dapi_labels)
    cond_bk, cond_bk_img = detect_background(condensate_img, dapi_labels_in, bk_th_method)

    # Save bk img for inspection
    bk_df = pd.DataFrame(cond_bk, columns=['z','mean_background_intensity'])
    imwrite(f"{output_dir}/{file_root}-ch{condensate_channel}-BK.tif", cond_bk_img.astype('uint8')*255, 
            imagej=True, metadata={'axes': 'ZYX'})
    
    # clear those touching border
    if clear_border:
        dapi_labels = segmentation.clear_border(dapi_labels)
    
    # output checking image in output dir with matched name
    ann_img = make_annotated_stack(condensate_img, lmask_condensates, dapi_labels, bit_depth=bit_depth)
    imwrite(f"{output_dir}/{file_root}-ANN.tif", ann_img, imagej=True, metadata={'axes': 'ZCYX'})

    # Gather spot information for each nuclei
    # Converting Area from px to um based on area calculated by skimage will not be correct 
    # z conversion is different from x/y conversion of px to um
    # If we know how many z-levels the object covers, can we adjust it correctly?
    nucl_df = pd.DataFrame(measure.regionprops_table(dapi_labels, dapi_img, 
                                                     properties=['label', 
                                                                 'area', 
                                                                 'mean_intensity', 
                                                                 'coords',
                                                                 'centroid',
                                                                'bbox']))
    foci_counts = []
    for row in nucl_df.iterrows():
        coords = row[1]['coords']
        zz, rr, cc = list(zip(*coords))

        # mask out any spots outside current nucleus, then get region props
        mult_img = np.zeros_like(lmask_condensates)
        mult_img[zz, rr, cc] = 1
        nucleus_spot_labels = lmask_condensates * mult_img

        foci_df = pd.DataFrame(measure.regionprops_table(nucleus_spot_labels,
                                                         condensate_img,
                                                         properties=['label',
                                                                     'area',
                                                                     'mean_intensity',
                                                                     'intensity_image',
                                                                     'centroid',
                                                                     'bbox'
                                                                     ]))
        # Compute total intensity
        total_intensities = [np.sum(arr) for arr in foci_df['intensity_image']]
        foci_df['total_intensity'] = total_intensities
        foci_df.drop(columns='intensity_image', inplace=True)  # optional cleanup

        # Save the spot areas + count for each nucleus, and add to main data frame
        if len(foci_df) > 0:
            foci_df['file'] = file_root
            foci_df['nucleus_label'] = row[1]['label']
            foci_df['count'] = len(foci_df)

            # for each foci, get min/max zlevel.  get background over those z levels and take mean
            # then add as a column - "background_intensity"
            bks=[]
            areas=[]
            for foci_row in foci_df.iterrows():
                zstart = foci_row[1]['bbox-0']
                zend = foci_row[1]['bbox-3']
                bks.append(np.mean(bk_df[(bk_df['z'] >= zstart) & (bk_df['z'] <= zend)]['mean_background_intensity']))
                areas.append(foci_row[1]['area']*XY_micron_per_px*XY_micron_per_px*Z_micron_per_px*(zend-zstart+1))
            foci_df['background_mean_intensity']=bks
            foci_df['volume_um']=areas

            # Compute background-subtracted total intensity
            foci_df['background_subtracted_total_intensity'] = (
                foci_df['total_intensity'] - foci_df['background_mean_intensity'] * foci_df['area']
            )

            # Concat to final df:
            foci_data = pd.concat([foci_data, foci_df], axis=0, ignore_index=True)

        # Save the foci count separately to join with the nucleus df
        foci_counts.append([row[1]['label'], len(foci_df)])

    # add foci counts to the nucleus df
    foci_counts_df = pd.DataFrame(foci_counts, columns=['label', 'foci_count'])
    nucl_df = nucl_df.join(foci_counts_df.set_index('label'), on='label', how='left')

    # add file and fov to the nucleus df, drop unwanted columns and concat to full nucleus df
    nucl_df['file'] = file_root
    nucl_df.drop(['coords', ], axis=1, inplace=True)
    nucleus_data = pd.concat([nucleus_data, nucl_df], axis=0, ignore_index=True)

    # Add file to bk df and concat
    bk_df['file']=file_root
    bk_data = pd.concat([bk_data, bk_df], axis=0, ignore_index=True)

# Save the output data frames
foci_data.to_csv(f"{output_dir}/foci_data.txt", sep='\t')
nucleus_data.to_csv(f"{output_dir}/nucleus_data.txt", sep='\t')
bk_data.to_csv(f"{output_dir}/ch{condensate_channel}_background.txt", sep='\t')